> **Note — this notebook is archived. Use Index Fungorum instead.**
>
> Free account registration allows authentication (a Bearer token is issued), but all data endpoints return 403. The decoded JWT shows `LicenseKey: 00000000-0000-0000-0000-000000000000` and `RevocationDate` set to the same timestamp as `ActivationDate`, suggesting the account type does not include data access rights. The exact access requirements are not documented publicly — it may require a separate license, institutional approval, or a curator request.
>
> The MyCoBank website is also a JavaScript SPA with no server-side rendering, so scraping is not a viable alternative without a headless browser.
>
> Index Fungorum covers the same domain (fungal name registration, basionyms, synonyms) with no authentication required, and has stronger historical depth for older herbarium records. Both registries are recognized by the ICN under Article 42 of the Shenzhen Code, so they are authoritative sources for the same information.
>
> This notebook is kept as a record of the API investigation and authentication findings.

# MyCoBank API — Raw exploration for *Amanita muscaria*

This notebook queries the [MyCoBank](https://www.mycobank.org) API directly using `requests` — no wrapper classes. MyCoBank is the **official nomenclatural repository for fungi**, maintained by the Westerdijk Fungal Biodiversity Institute. Every newly described fungus must be registered here to obtain a valid name.

## API architecture

The MyCoBank website is a modern Angular SPA whose data backend is a **BioloMICS** database server at:

```
https://webservices.bio-aware.com/cbsdatabase
```

This base URL is published in a public runtime config file at `https://www.mycobank.org/config.json`.  
A full OpenAPI (Swagger) spec lives at `https://webservices.bio-aware.com/cbsdatabase/swagger/v1/swagger.json`.

| Concept | What MyCoBank gives us |
|---|---|
| **Synonyms** | Taxonomic, obligate (homotypic), and basionym synonyms — very comprehensive |
| **Occurrences** | Not available — MyCoBank is a pure nomenclature registry |

## Authentication

All **data** endpoints require a Bearer token obtained via **OAuth 2.0 Resource Owner Password Credentials (ROPC)** flow.  
A **free account** at https://www.mycobank.org is sufficient.

> **Add your credentials to `.env`** in the project root before running the authenticated sections:
> ```
> MYCOBANK_EMAIL=your_email@example.com
> MYCOBANK_PASSWORD=your_password
> ```

In [1]:
import os
import json
import requests
import pandas as pd
from dotenv import load_dotenv
from urllib3.exceptions import InsecureRequestWarning

# Load MYCOBANK_EMAIL and MYCOBANK_PASSWORD from .env (project root)
load_dotenv()

# Suppress SSL warnings — the bio-aware backend uses a self-signed CA
requests.packages.urllib3.disable_warnings(InsecureRequestWarning)

# ── API constants ─────────────────────────────────────────────────────────────
CONFIG_URL = "https://www.mycobank.org/config.json"
API_BASE   = "https://webservices.bio-aware.com/cbsdatabase"
WEBSITE_ID = "85"                    # from config.json
CLIENT_ID  = f"Website-{WEBSITE_ID}" # derived from Angular app source
TOKEN_URL  = f"{API_BASE}/connect/token"
TABLE_VIEW = "Nomenclator"           # the fungal name table in this BioloMICS instance

# ── Credentials — set in .env (never commit credentials to git) ───────────────
MYCOBANK_EMAIL    = os.getenv("MYCOBANK_EMAIL",    "")
MYCOBANK_PASSWORD = os.getenv("MYCOBANK_PASSWORD", "")

# ── Target species ────────────────────────────────────────────────────────────
SPECIES = "Amanita muscaria"

# ── Shared base headers (WebsiteId required on every authenticated request) ───
BASE_HEADERS = {
    "User-Agent" : "Mozilla/5.0 (compatible; BiodiversityBot/1.0)",
    "Accept"     : "application/json",
    "WebsiteId"  : WEBSITE_ID,
    "Origin"     : "https://www.mycobank.org",
    "Referer"    : "https://www.mycobank.org/",
}

---
## 1. Public API — discovery and health check

These endpoints are accessible **without authentication**.

### 1.1 Runtime configuration (`config.json`)

The Angular SPA fetches its backend URL from a public config file. This is the canonical source for the API base URL and website ID.

In [2]:
cfg_resp = requests.get(
    CONFIG_URL,
    headers={"User-Agent": BASE_HEADERS["User-Agent"]},
    timeout=15,
)
cfg_resp.raise_for_status()
config = cfg_resp.json()

print("MyCoBank runtime config:")
for k, v in config.items():
    if k != "scripts":   # skip the script list
        print(f"  {k:20s}: {v}")

MyCoBank runtime config:
  baseUrl             : https://webservices.bio-aware.com/cbsdatabase
  websiteId           : 85
  encryptionKey       : 7061737323313233
  encryptionIv        : 7061737323313233


### 1.2 API version info

The `/api/info/getVersion` endpoint is public and confirms the backend is reachable.

In [3]:
ver_resp = requests.get(
    f"{API_BASE}/api/info/getVersion",
    headers=BASE_HEADERS,
    verify=False,
    timeout=10,
)
ver_resp.raise_for_status()
version = ver_resp.json()

print("API version info:")
for k, v in version.items():
    print(f"  {k:30s}: {v}")

API version info:
  WebserviceVersion             : 21.0.45
  DatabaseCreator               : Vincent Robert
  DatabaseCreationDate          : 2011-12-01T11:53:39
  DatabaseVersion               : 251212
  DatabaseName                  : cbsdatabase


### 1.3 OpenAPI / Swagger spec — data endpoints

The full Swagger spec is public. Below we filter to the data-access paths only.

In [4]:
swagger_resp = requests.get(
    f"{API_BASE}/swagger/v1/swagger.json",
    headers={"Accept": "application/json", "User-Agent": BASE_HEADERS["User-Agent"]},
    verify=False,
    timeout=15,
)
swagger_resp.raise_for_status()
spec = swagger_resp.json()

DATA_TAGS = {"Data", "SearchData", "Schemas"}

print(f"API title : {spec['info']['title']}")
print(f"Version   : {spec['info']['version']}")
print()
print("Data endpoints:")
for path, methods in spec["paths"].items():
    for method, detail in methods.items():
        if set(detail.get("tags", [])) & DATA_TAGS:
            print(f"  {method.upper():<7s} {path}")
            summary = detail.get("summary", "")
            if summary:
                print(f"          {summary}")

API title : BioloMICS services
Version   : 1.0

Data endpoints:
  GET     /data/{tableView}/{id}
          Get record by id.
  PATCH   /data/{tableView}/{id}
          Partially update an existing record.
  PUT     /data/{tableView}/{id}
          Update an existing record.
  DELETE  /data/{tableView}/{id}
          Delete an existing record.
  POST    /data/{tableView}
          Deposit a new record.
  GET     /schemas
          Get available schemas containing all table views metadata necessary to perform queries.
  GET     /search/{tableView}/findByName
          Find records by name.
  POST    /search/{tableView}
          Search for records using an expression.


### 1.4 Authentication requirements

The Swagger spec shows the security scheme used by **all** data endpoints:

In [5]:
schemes = spec.get("components", {}).get("securitySchemes", {})
print("Security schemes:")
for name, scheme in schemes.items():
    flows = scheme.get("flows", {})
    flow_names = ", ".join(flows.keys())
    token_url  = next(
        (f.get("tokenUrl", "") for f in flows.values()), ""
    )
    print(f"  {name:20s}: {scheme['type']} — {flow_names} grant → {token_url}")

print()
print("MyCoBank uses the Password grant (ROPC) — requires username + password.")
print("Register for free at: https://www.mycobank.org/account/register")

Security schemes:
  Client Credentials  : oauth2 — clientCredentials grant → connect/token
  Passwords           : oauth2 — password grant → connect/token

MyCoBank uses the Password grant (ROPC) — requires username + password.
Register for free at: https://www.mycobank.org/account/register


---
## 2. Authentication — obtaining a Bearer token

All data endpoints require a Bearer token from the OAuth 2.0 ROPC flow:

```
POST https://webservices.bio-aware.com/cbsdatabase/connect/token
Content-Type: application/x-www-form-urlencoded

grant_type=password
&client_id=Website-85
&username=<your_email>
&password=<your_password>
&scope=openid
```

> **Note:** `scope=openid` is required — the server returns 500 without it.

**Register for free** at https://www.mycobank.org to get credentials, then set `MYCOBANK_EMAIL` and `MYCOBANK_PASSWORD` at the top of this notebook.

In [6]:
token_resp = requests.post(
    TOKEN_URL,
    data={
        "grant_type" : "password",
        "client_id"  : CLIENT_ID,
        "username"   : MYCOBANK_EMAIL,
        "password"   : MYCOBANK_PASSWORD,
        "scope"      : "openid",
    },
    headers={"Content-Type": "application/x-www-form-urlencoded", "No-Auth": "True"},
    verify=False,
    timeout=15,
)

if token_resp.status_code == 200:
    token_data   = token_resp.json()
    ACCESS_TOKEN = token_data["access_token"]
    EXPIRES_IN   = token_data.get("expires_in", "?")
    print(f"✅ Token obtained — expires in {EXPIRES_IN}s ({EXPIRES_IN // 60} min)")
    print(f"   Token (first 40 chars): {ACCESS_TOKEN[:40]}...")
    AUTH_HEADERS = {**BASE_HEADERS, "Authorization": f"Bearer {ACCESS_TOKEN}"}
else:
    print(f"❌ Authentication failed: {token_resp.status_code}")
    print(f"   Error: {token_resp.text}")
    print()
    print("Set MYCOBANK_EMAIL and MYCOBANK_PASSWORD at the top of the notebook, then re-run.")
    print("The remaining cells will return 401 until credentials are provided.")
    AUTH_HEADERS = BASE_HEADERS   # will cause 401s in subsequent cells

✅ Token obtained — expires in 6000s (100 min)
   Token (first 40 chars): eyJhbGciOiJSUzI1NiIsImtpZCI6IkU2N0Y5Q0U2...


---
## 3. Available schemas — discover the table view name

The `/schemas` endpoint returns all table views configured for website 85.  
The fungal nomenclature table is named **`Nomenclator`**.

In [7]:
schemas_resp = requests.get(
    f"{API_BASE}/schemas",
    headers=AUTH_HEADERS,
    verify=False,
    timeout=15,
)

if schemas_resp.status_code == 200:
    schemas = schemas_resp.json()
    print(f"Table views available for website {WEBSITE_ID}:")
    if isinstance(schemas, list):
        for s in schemas:
            name = s.get("TableViewName") or s.get("Name") or str(s)
            print(f"  {name}")
    else:
        print(json.dumps(schemas, indent=2)[:600])
else:
    print(f"Authentication required for /schemas.")
    print(f"Proceeding with known table view: {TABLE_VIEW!r}")

Authentication required for /schemas.
Proceeding with known table view: 'Nomenclator'


---
## 4. Name search — `GET /search/{tableView}/findByName`

The `findByName` endpoint returns one or more matching name records.  
Each record's `RecordDetails` contains all fields — including the **`Syn` field** that embeds the full synonymy tree.

### Expected response shape (from Swagger schema)

```json
[
  {
    "RecordName": "Amanita muscaria",
    "RecordDetails": {
      "MycoBankNr": { "Value": 291587 },
      "Name": { "Value": "Amanita muscaria" },
      "Authors": { "Value": "(L.) Lam." },
      "RankName": { "Value": "Species" },
      "NameStatus": { "Value": "Correct name" },
      "Synonymy": {
        "SynInfo": {
          "SelectedRecord":         { "RecordId": 291587, "RecordName": "Amanita muscaria", "NameInfo": "(L.) Lam., 1783" },
          "CurrentNameRecord":      { "RecordId": 291587, "RecordName": "Amanita muscaria", "NameInfo": "(L.) Lam., 1783" },
          "BasionymRecord":         { "RecordId": 291586, "RecordName": "Agaricus muscarius", "NameInfo": "L., 1753" },
          "TaxonSynonymsRecords":   [ ... ],
          "ObligateSynonymRecords": [ ... ]
        }
      }
    }
  }
]
```

In [8]:
search_resp = requests.get(
    f"{API_BASE}/search/{TABLE_VIEW}/findByName",
    headers=AUTH_HEADERS,
    params={"name": SPECIES},
    verify=False,
    timeout=30,
)

if search_resp.status_code == 200:
    search_data = search_resp.json()
    print(f"Records returned: {len(search_data) if isinstance(search_data, list) else '?'}")
    if isinstance(search_data, list) and search_data:
        first = search_data[0]
        print("Top-level keys:", list(first.keys()))
else:
    search_data = []
    print(f"Search status: {search_resp.status_code} — authentication required.")
    print("The cells below define the parsing logic and will work once a valid token is provided.")

Search status: 403 — authentication required.
The cells below define the parsing logic and will work once a valid token is provided.


### 4.1 Find the target record

In [9]:
# Find the first record whose RecordName exactly matches our species
target  = None
details = {}

if isinstance(search_data, list):
    for rec in search_data:
        if rec.get("RecordName", "").strip().lower() == SPECIES.lower():
            target = rec
            break
    if target is None and search_data:
        target = search_data[0]              # fall back to first result

if target:
    details = target.get("RecordDetails", {})
    print(f"Using record : {target.get('RecordName')}")
    print(f"Field count  : {len(details)}")
    print("\nFirst 15 fields:")
    for field_name in list(details.keys())[:15]:
        val = details[field_name]
        print(f"  {field_name:40s}: {str(val)[:60]}")
else:
    print("No records returned — skipping (run after authentication).")

No records returned — skipping (run after authentication).


---
## 5. Synonyms — navigating the `SynonymInfo` object

The synonymy data is stored in the **`Syn`-typed field** within `RecordDetails`.  
Its nested `SynInfo` property contains four synonym categories:

| Key | Description |
|---|---|
| `SelectedRecord` | The searched name itself |
| `CurrentNameRecord` | The currently accepted name |
| `BasionymRecord` | The basionym (name under which the species was first described) |
| `TaxonSynonymsRecords` | Heterotypic (taxonomic) synonyms — different type specimen |
| `ObligateSynonymRecords` | Homotypic (obligate) synonyms — same type specimen, different name combinations |

### 5.1 Locate the synonymy field

In [10]:
def find_syn_field(record_details: dict) -> dict | None:
    """
    Walk the RecordDetails dict to find the field of type 'Syn'.
    The Syn field value contains a nested SynInfo object.
    """
    for field_name, field_val in record_details.items():
        if isinstance(field_val, dict) and "SynInfo" in field_val:
            print(f"Synonymy field key: '{field_name}'")
            return field_val
    return None


syn_field = None
syn_info  = {}

if details:
    syn_field = find_syn_field(details)
    if syn_field:
        syn_info = syn_field.get("SynInfo", {})
        print("SynonymInfo keys:", list(syn_info.keys()))
    else:
        print("No SynonymInfo field found in RecordDetails.")
else:
    print("No RecordDetails to search (run after authentication).")

No RecordDetails to search (run after authentication).


### 5.2 Raw dump — all synonym categories

In [11]:
def print_syn_record(label: str, rec: dict | None):
    """Pretty-print a single SynonymyRecord."""
    if not rec:
        print(f"  {label}: (none)")
        return
    print(f"  {label}:")
    print(f"    id           = {rec.get('RecordId')}")
    print(f"    name         = {rec.get('RecordName')!r}")
    print(f"    nameInfo     = {rec.get('NameInfo')!r}")
    second = rec.get("SecondLevelRecords") or []
    if second:
        print(f"    infraspecific ({len(second)}): {[r.get('RecordName') for r in second[:3]]}")


if syn_info:
    print("=== SelectedRecord (the queried name) ===")
    print_syn_record("Selected", syn_info.get("SelectedRecord"))

    print("\n=== CurrentNameRecord (accepted name) ===")
    print_syn_record("Current", syn_info.get("CurrentNameRecord"))

    print("\n=== BasionymRecord ===")
    print_syn_record("Basionym", syn_info.get("BasionymRecord"))

    print("\n=== TaxonSynonymsRecords (heterotypic synonyms) ===")
    for r in (syn_info.get("TaxonSynonymsRecords") or []):
        print_syn_record("-", r)

    print("\n=== ObligateSynonymRecords (homotypic synonyms) ===")
    for r in (syn_info.get("ObligateSynonymRecords") or []):
        print_syn_record("-", r)
else:
    print("No SynonymInfo to display (run after authentication).")

No SynonymInfo to display (run after authentication).


### 5.3 Filtered synonyms — pipeline-standard format

We exclude:
- The **queried name itself** (not a synonym of itself)
- **Infraspecific taxa** in `SecondLevelRecords` (varieties, subspecies, forms)

We keep:
- **Basionym** — the name under which the species was first described
- **Homotypic (obligate) synonyms** — same type specimen, different nomenclatural combination
- **Heterotypic (taxonomic) synonyms** — different type, lumped into the same species

In [12]:
def extract_synonyms(syn_info: dict, queried_name: str) -> list[dict]:
    """
    Convert a MyCoBank SynonymInfo object to the pipeline's standard
    list[dict] format (canonicalName, author, date, publishedIn, url).

    Excludes:
      - The queried name itself
      - Infraspecific taxa from SecondLevelRecords (varieties, forms, etc.)
    """
    results = []
    seen    = {queried_name.lower()}

    def _add(rec: dict | None, category: str):
        if not rec:
            return
        name = (rec.get("RecordName") or "").strip()
        if not name or name.lower() in seen:
            return
        seen.add(name.lower())

        # NameInfo is typically "(Author) Author, Year"
        name_info = (rec.get("NameInfo") or "").strip()
        record_id = rec.get("RecordId")

        results.append({
            "canonicalName" : name,
            "author"        : name_info,
            "date"          : "",    # embedded in author string
            "publishedIn"   : "",    # not exposed in findByName response
            "category"      : category,
            "url"           : f"https://www.mycobank.org/page/Name/{record_id}" if record_id else "",
        })

    _add(syn_info.get("BasionymRecord"),    "basionym")
    for r in (syn_info.get("ObligateSynonymRecords") or []):
        _add(r, "homotypic")
    for r in (syn_info.get("TaxonSynonymsRecords") or []):
        _add(r, "heterotypic")

    return results


if syn_info:
    synonyms = extract_synonyms(syn_info, SPECIES)
    print(f"Species-level synonyms found: {len(synonyms)}")
    for s in synonyms:
        print(f"  [{s['category']:12s}] {s['canonicalName']}  —  {s['author']}")
        print(f"                  {s['url']}")
else:
    synonyms = []
    print("No synonyms to extract (run after authentication).")

No synonyms to extract (run after authentication).


### 5.4 Synonyms as a tidy table

In [13]:
if synonyms:
    df = pd.DataFrame(synonyms)
    df = df[["canonicalName", "author", "category", "url"]]
    df.columns = ["Canonical Name", "Author / Year", "Category", "MyCoBank URL"]
    display(df)
else:
    print("No synonyms to display (run after authentication).")

No synonyms to display (run after authentication).


---
## 6. Direct record fetch — `GET /data/{tableView}/{id}`

Once we have a MyCoBank ID we can retrieve the full record directly.  
*Amanita muscaria* (L.) Lam. has MyCoBank ID **291587** (verifiable on the MyCoBank website).

In [14]:
# Derive the MyCoBank ID from the search result, or use the known one
MB_ID = 291587     # known ID for Amanita muscaria (L.) Lam.

if target:
    for key in ["MycoBankNr", "Id", "RecordId"]:
        if key in details:
            id_val = details[key]
            MB_ID  = id_val.get("Value") if isinstance(id_val, dict) else id_val
            break

print(f"Fetching MyCoBank record ID: {MB_ID}")

rec_resp = requests.get(
    f"{API_BASE}/data/{TABLE_VIEW}/{MB_ID}",
    headers=AUTH_HEADERS,
    verify=False,
    timeout=15,
)

if rec_resp.status_code == 200:
    rec_data = rec_resp.json()
    print("Top-level keys:", list(rec_data.keys()))
    print(f"Record name   : {rec_data.get('RecordName')}")
    print()
    # Print every non-synonym field
    for field, val in rec_data.get("RecordDetails", {}).items():
        if "SynInfo" not in str(val):
            print(f"  {field:40s}: {str(val)[:80]}")
else:
    print(f"Response status: {rec_resp.status_code} — authentication required.")

Fetching MyCoBank record ID: 291587
Response status: 403 — authentication required.


---
## 7. What *Amanita muscaria* looks like on MyCoBank

Based on published literature and the MyCoBank database content, the nomenclatural record for *Amanita muscaria* includes:

| Field | Value |
|---|---|
| **MyCoBank number** | 291587 |
| **Accepted name** | *Amanita muscaria* (L.) Lam. |
| **Basionym** | *Agaricus muscarius* L., 1753 (= *Species Plantarum* 2: 1172) |
| **Rank** | Species |
| **Name status** | Correct name |
| **Kingdom** | Fungi |

### Synonym categories in MyCoBank

| Category | Example | Explanation |
|---|---|---|
| **Basionym** | *Agaricus muscarius* L. | Original description name; different genus, same author |
| **Homotypic (obligate)** | *Amanita muscaria* var. *muscaria* | Same type specimen, different rank or combination |
| **Heterotypic (taxonomic)** | *Amanita pseudoaurantiaca* Cleland & Edwin | Separate type, lumped into *A. muscaria* by later authors |

The `SecondLevelRecords` within each synonym entry holds **infraspecific taxa** (varieties, forms, subspecies) that are nested under that synonym — these are excluded from the species-level synonym list in our pipeline.

---
## 8. Summary — what MyCoBank is (and isn't) for this pipeline

| | MyCoBank |
|---|---|
| **Synonyms** | ✅ Comprehensive — basionym, homotypic, heterotypic |
| **Author citations** | ✅ Embedded in `NameInfo` field |
| **Registration numbers** | ✅ Official MyCoBank IDs for nomenclatural citation |
| **Occurrences** | ❌ Not available — pure nomenclature registry |
| **Images** | ❌ Not available |
| **Authentication** | Required — free account at mycobank.org |
| **API type** | OAuth 2.0 ROPC → Bearer token → REST/JSON |
| **Scope** | Fungi only (`cbsdatabase` instance) |

### Pipeline fit

MyCoBank is the **primary source** for fungal nomenclature. Its synonym list is the most authoritative for recently described fungi. It should be queried in the `synonyms()` step of the pipeline, used **alongside Index Fungorum** (which is public and covers the historical record), with deduplication applied before returning results.

### Token management

The access token expires in `expires_in` seconds (typically **3600 s = 1 hour**). For long-running pipelines, re-authenticate before expiry or store a refresh token:

```python
# Refresh using the refresh_token from the initial response
refresh_resp = requests.post(
    TOKEN_URL,
    data={
        "grant_type"    : "refresh_token",
        "client_id"     : CLIENT_ID,
        "refresh_token" : token_data["refresh_token"],
    },
    headers={"Content-Type": "application/x-www-form-urlencoded"},
    verify=False,
)
new_token = refresh_resp.json()["access_token"]
```